In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import sys
sys.path.append('..')

from label2id import label2id_dict

In [2]:
# Load model directly
from transformers import AutoImageProcessor, AutoModelForImageClassification, pipeline
from datasets import load_dataset

import tqdm

import torch
from torch.utils.data import DataLoader

processor = AutoImageProcessor.from_pretrained("nateraw/vit-base-food101")
model = AutoModelForImageClassification.from_pretrained("nateraw/vit-base-food101")
model.eval()

preprocessor_config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.93k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/344M [00:00<?, ?B/s]

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=7

In [3]:
print(model)

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=7

In [4]:
dataset = load_dataset("food101")['validation']
print(dataset)

Dataset({
    features: ['image', 'label'],
    num_rows: 25250
})


In [12]:
classifier = pipeline("image-classification", model=model, image_processor=processor, device='cuda')
# examples from training set

img = dataset[0]['image'] # beignets 6

print(label2id_dict[classifier(img)[0]['label']])
print(dataset[0]['label'])

img = dataset[-1]['image'] # spaghetti_bolognese 90

print(label2id_dict[classifier(img)[0]['label']])
print(dataset[-1]['label'])

6
6
90
90


In [16]:
# 定义预处理函数
def process_dataset(example_batch):
    # Take a list of PIL images and turn them to pixel values
    inputs = processor([x.convert('RGB') for x in example_batch['image']], return_tensors='pt')

    # Don't forget to include the labels!
    inputs['label'] = example_batch['label']
    return inputs

dataset.set_transform(process_dataset)


In [19]:
batch_size = 192  # You can adjust this based on your GPU memory
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

correct = 0
total = 0

model = model.to('cuda')

with torch.no_grad():  # Disable gradient calculation for evaluation
    for batch in tqdm.tqdm(dataloader):
        pixel_values = batch['pixel_values'].to('cuda')
        labels = batch['label'].to('cuda')
        
        # Process images in batch
        batch_results = model(pixel_values=pixel_values)  # Assuming your classifier can handle batches
        
        predicted_labels = batch_results.logits.argmax(dim=-1)
        
        # Compare with ground truth
        correct += (predicted_labels == labels).sum().item()
        total += len(labels)

accuracy = correct / total
print(f"Correct: {correct}, Total: {total}, Accuracy: {accuracy:7f}")

100%|██████████| 132/132 [03:04<00:00,  1.40s/it]

Correct: 20592, Total: 25250, Accuracy: 0.815525
